<a href="https://colab.research.google.com/github/MwangiMuriuki2003/MURIUKI/blob/main/CCC_Hackathon_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pypdf
import pypdf

# Open and read the PDF file
reader = pypdf.PdfReader('/content/level-2.pdf')
print(f"Total pages: {len(reader.pages)}")

# Extract and display the text from each page
for i, page in enumerate(reader.pages):
    print(f"--- Page {i+1} ---")
    print(page.extract_text())
    print("\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 2.1 MB/s eta 0:00:00
Total pages: 3
--- Page 1 ---
Level 2
LEVEL 2
LEVEL 2#WeLoveSoftwareu too? cloudflight.io/career


--- Page 2 ---
Level 2: The Bird Love Score
So you now have a sorted list, great job!
But you remember, like the other expert Dr. Oldfeather
always said, “it’s not just temperature and humidity you 
fools!!1!”.
The urge of birds migrating to an area shall be called 
“Bird Love Score” and shall also consider vegetation, 
insects and urban light.
continue



--- Page 3 ---
Level 2: Acceptance Criteria
For this level to succeed, upload a csv file, containing 
the predictions for all “missing” Bird Love Scores.
Submission sample (included in level_2.zip):
You will need a good prediction to complete the level.
• RMSE < 6 for level_2_a
• RMSE < 3 for level_2_b
• RMSE < 3 for level_2_c
That’s it – good luck!
submit
BOP,Bird Love Score [<3]
3,28.20
4,42.95
Level 2: The Bird Love Score
continue
• You created a new tabl

In [3]:
import os

def preview_file(filepath, n_lines=15):
    print(f"=== Preview of {filepath} ===")
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            for i in range(n_lines):
                line = f.readline()
                if not line:
                    break
                print(line.strip())
    else:
        print("File not found!")
    print("\n")

# Preview the files
preview_file('/content/in_level-2_0-example.txt')
preview_file('/content/in_level-2_1-a.txt')
preview_file('/content/all_data_from_level_1.in')

=== Preview of /content/in_level-2_0-example.txt ===
BOP,Vegetation [%],Insects [g/m²],Urban Light [%],Bird Love Score [<3]
1,42.07,10.81,40.83,52.44
2,53.33,10.32,39.86,68.62
3,57.13,12.97,10.11,missing
4,59.45,12.30,67.18,missing


=== Preview of /content/in_level-2_1-a.txt ===
BOP,Vegetation [%],Insects [g/m²],Urban Light [%],Bird Love Score [<3]
4,59.45,12.30,67.18,42.95
6,49.27,12.50,23.18,76.06
8,34.91,13.46,13.45,44.83
10,62.26,9.11,23.88,84.95
13,30.19,6.92,13.11,33.36
16,47.30,13.47,67.78,59.75
17,54.81,13.49,46.01,45.57
30,47.38,10.78,69.09,60.21
31,57.30,10.38,45.05,42.45
34,45.69,13.60,27.73,29.20
37,36.91,10.78,85.41,72.87
39,34.93,10.71,89.21,56.12
41,65.30,7.87,7.45,22.45
42,58.17,13.46,36.91,87.39


=== Preview of /content/all_data_from_level_1.in ===
BOP,Temperature [°C],Humidity [%]
1,22,72
2,32,39
3,6,77
4,17,89
5,28,42
6,37,52
7,10,31
8,18,86
9,19,29
10,39,89
11,37,64
12,43,79
13,10,26
14,29,11




In [5]:
import glob
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingRegressor

# 1. Load and Clean Level 1 Data
df_lvl1 = pd.read_csv('/content/all_data_from_level_1.in')

def clean_temp(temp):
    if temp > 45:
        return round((temp - 32) * 5 / 9)
    return temp

df_lvl1['Temperature [°C]'] = df_lvl1['Temperature [°C]'].apply(clean_temp)

# 2. Find all Level 2 files
input_files = sorted(glob.glob('/content/in_level-2_*.txt'))
print(f"Found input files: {input_files}")

# Step 2a. Gather all training data from all files to build a global robust dataset
all_train_data = []

for infile in input_files:
    suffix = os.path.basename(infile).replace('in_level-2_', '').replace('.txt', '')
    if suffix == '0-example':
        continue

    df_lvl2 = pd.read_csv(infile)
    df_merged = pd.merge(df_lvl2, df_lvl1, on='BOP', how='left')
    df_merged['Bird Love Score [<3]'] = pd.to_numeric(df_merged['Bird Love Score [<3]'], errors='coerce')

    train_samples = df_merged[df_merged['Bird Love Score [<3]'].notna()]
    if len(train_samples) > 0:
        all_train_data.append(train_samples)

df_all_train = pd.concat(all_train_data, ignore_index=True)
print(f"Combined training samples: {len(df_all_train)}")

feature_cols = ['Vegetation [%]', 'Insects [g/m²]', 'Urban Light [%]', 'Temperature [°C]', 'Humidity [%]']
X_train_global = df_all_train[feature_cols]
y_train_global = df_all_train['Bird Love Score [<3]']

# Train a highly accurate global model
global_model = GradientBoostingRegressor(n_estimators=250, max_depth=5, learning_rate=0.05, random_state=42)
global_model.fit(X_train_global, y_train_global)

# 3. Predict missing values for each individual file using the global model
for infile in input_files:
    suffix = os.path.basename(infile).replace('in_level-2_', '').replace('.txt', '')
    if suffix == '0-example':
        continue

    print(f"\nPredicting for {infile} (Suffix: {suffix})...")

    df_lvl2 = pd.read_csv(infile)
    df_merged = pd.merge(df_lvl2, df_lvl1, on='BOP', how='left')
    df_merged['Bird Love Score [<3]'] = pd.to_numeric(df_merged['Bird Love Score [<3]'], errors='coerce')

    # Identify rows that need prediction
    predict_data = df_merged[df_merged['Bird Love Score [<3]'].isna()]
    print(f"Predict samples: {len(predict_data)}")

    if len(predict_data) > 0:
        X_predict = predict_data[feature_cols]
        predictions = global_model.predict(X_predict)

        df_output = pd.DataFrame({
            'BOP': predict_data['BOP'],
            'Bird Love Score [<3]': np.round(predictions, 2)
        })

        out_filename = f'/content/out_level-2_{suffix}.csv'
        df_output.to_csv(out_filename, index=False)
        print(f"Successfully saved predictions to {out_filename}")
        print(df_output.head())

Found input files: ['/content/in_level-2_0-example.txt', '/content/in_level-2_1-a.txt', '/content/in_level-2_2-b.txt', '/content/in_level-2_3-c.txt']
Combined training samples: 2000

Predicting for /content/in_level-2_1-a.txt (Suffix: 1-a)...
Predict samples: 200
Successfully saved predictions to /content/out_level-2_1-a.csv
     BOP  Bird Love Score [<3]
800    2                 68.18
801    3                 24.63
802   19                 70.60
803   22                 75.30
804   24                 78.17

Predicting for /content/in_level-2_2-b.txt (Suffix: 2-b)...
Predict samples: 200
Successfully saved predictions to /content/out_level-2_2-b.csv
      BOP  Bird Love Score [<3]
1200    1                 52.65
1201   25                 69.87
1202   29                  8.26
1203   35                 75.62
1204   47                 68.03

Predicting for /content/in_level-2_3-c.txt (Suffix: 3-c)...
Predict samples: 100
Successfully saved predictions to /content/out_level-2_3-c.csv
   BO